In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [2]:
text = """
the cat sat on the mat
the dog sat on the rug
the cat chased the mouse
the dog chased the cat
"""

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

In [4]:
print(tokenizer.word_index)

{'the': 1, 'cat': 2, 'sat': 3, 'on': 4, 'dog': 5, 'chased': 6, 'mat': 7, 'rug': 8, 'mouse': 9}


In [5]:
total_words = len(tokenizer.word_index)

In [6]:
total_words

9

In [7]:
input_sequences =[]
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1,len(token_list)):
      n_gram_sequence = token_list[:i+1]
      input_sequences.append(n_gram_sequence)

In [8]:
max_sequence_len = max(len(sequence) for sequence in input_sequences)

input_sequences = pad_sequences(input_sequences,maxlen=max_sequence_len,padding='pre')

In [9]:
input_sequences


array([[0, 0, 0, 0, 1, 2],
       [0, 0, 0, 1, 2, 3],
       [0, 0, 1, 2, 3, 4],
       [0, 1, 2, 3, 4, 1],
       [1, 2, 3, 4, 1, 7],
       [0, 0, 0, 0, 1, 5],
       [0, 0, 0, 1, 5, 3],
       [0, 0, 1, 5, 3, 4],
       [0, 1, 5, 3, 4, 1],
       [1, 5, 3, 4, 1, 8],
       [0, 0, 0, 0, 1, 2],
       [0, 0, 0, 1, 2, 6],
       [0, 0, 1, 2, 6, 1],
       [0, 1, 2, 6, 1, 9],
       [0, 0, 0, 0, 1, 5],
       [0, 0, 0, 1, 5, 6],
       [0, 0, 1, 5, 6, 1],
       [0, 1, 5, 6, 1, 2]], dtype=int32)

In [10]:
X = input_sequences[:,:-1]
y = input_sequences[:,-1]

In [11]:
X.shape

(18, 5)

In [12]:
y.shape

(18,)

In [13]:
embedding_dim = 32

embedding_layer = tf.keras.layers.Embedding(input_dim = total_words, output_dim = embedding_dim)

In [14]:
embedded_X = embedding_layer(X)
print(embedded_X.shape)

(18, 5, 32)


In [15]:
class CustomLSTMCell(tf.keras.layers.Layer):

    def __init__(self, hidden_units):
        super().__init__()

        self.hidden_units = hidden_units


    def build(self, input_shape):

        input_dim = input_shape[-1]

        # Forget Gate
        self.W_f = self.add_weight(
            shape=(input_dim + self.hidden_units,
                   self.hidden_units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b_f = self.add_weight(
            shape=(self.hidden_units,),
            initializer="zeros",
            trainable=True
        )


        # Input Gate
        self.W_i = self.add_weight(
            shape=(input_dim + self.hidden_units,
                   self.hidden_units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b_i = self.add_weight(
            shape=(self.hidden_units,),
            initializer="zeros",
            trainable=True
        )


        # Candidate Cell State
        self.W_c = self.add_weight(
            shape=(input_dim + self.hidden_units,
                   self.hidden_units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b_c = self.add_weight(
            shape=(self.hidden_units,),
            initializer="zeros",
            trainable=True
        )


        # Output Gate
        self.W_o = self.add_weight(
            shape=(input_dim + self.hidden_units,
                   self.hidden_units),
            initializer="glorot_uniform",
            trainable=True
        )

        self.b_o = self.add_weight(
            shape=(self.hidden_units,),
            initializer="zeros",
            trainable=True
        )


_IncompleteInputError: incomplete input (795245362.py, line 4)

In [ ]:
def call(self, x_t, states):

    h_prev, c_prev = states

    # Combine current input and previous hidden state
    combined = tf.concat(
        [x_t, h_prev],
        axis=1
    )


    # Forget Gate
    f_t = tf.sigmoid(
        tf.matmul(combined, self.W_f) + self.b_f
    )


    # Input Gate
    i_t = tf.sigmoid(
        tf.matmul(combined, self.W_i) + self.b_i
    )


    # Candidate Cell State
    c_candidate = tf.tanh(
        tf.matmul(combined, self.W_c) + self.b_c
    )


    # Update Cell State
    c_t = (
        f_t * c_prev
        +
        i_t * c_candidate
    )


    # Output Gate
    o_t = tf.sigmoid(
        tf.matmul(combined, self.W_o) + self.b_o
    )


    # Update Hidden State
    h_t = o_t * tf.tanh(c_t)


    return h_t, [h_t, c_t]

In [ ]:
class CustomLSTM(tf.keras.layers.Layer):

    def __init__(self, hidden_units):
        super().__init__()

        self.hidden_units = hidden_units

        self.lstm_cell = CustomLSTMCell(hidden_units)


    def call(self, inputs):

        batch_size = tf.shape(inputs)[0]

        sequence_length = inputs.shape[1]


        # Initial Hidden State
        h = tf.zeros(
            (batch_size, self.hidden_units)
        )


        # Initial Cell State
        c = tf.zeros(
            (batch_size, self.hidden_units)
        )


        # Process each timestep
        for t in range(sequence_length):

            x_t = inputs[:, t, :]

            h, [h, c] = self.lstm_cell(
                x_t,
                [h, c]
            )


        return h

In [16]:
model = tf.keras.Sequential([

    tf.keras.layers.Embedding(
        input_dim=total_words,
        output_dim=embedding_dim
    ),

    CustomLSTM(
        hidden_units=64
    ),

    tf.keras.layers.Dense(
        total_words,
        activation="softmax"
    )
])

NameError: name 'CustomLSTM' is not defined